### Exercise 5.4
After saving the weights, load the model and optimizer in a new Python session or Jupyter notebook file and continue pretraining it for one more epoch using the train_model_simple function.  
(See Section 5.4)

> **Attribution:** Portions of the code in this notebook follow and adapt the Apache-2.0-licensed implementation accompanying Sebastian Raschka's *Build a Large Language Model (From Scratch)*.  
> Original source code: https://github.com/rasbt/LLMs-from-scratch  
> Additional annotations, experiments, explanations, and study notes were created as part of my own learning and implementation process.

In [1]:
# # Adding the repo root to sys.path
# from pathlib import Path
# import sys

# # Current folder:
# # repository_root/chapter_04/exercises
# # i.e. 
# # import os  
# # print(os.getcwd()) 
# # Note: Python searches for chapter_03 (and all other needed imports) inside that folder and in the other locations listed in sys.path. but sys.path currently does not have the repo root
# # the snippet below adds the repo root to sys.path

# repo_root = Path.cwd().parents[1]

# if str(repo_root) not in sys.path:
#     sys.path.insert(0, str(repo_root))

# print("Repository root:", repo_root)
# sys.path

In [ ]:
# Needed setup steps
import torch
import tiktoken
from chapter_04.scripts.gpt import GPTModel, GPT2_cfg_small
from chapter_05.scripts.training import train_model_simple

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from chapter_02.scripts.data_pipeline import create_dataloader_v1
torch.manual_seed(123)

file_path = "../../chapter_02/data/the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as file:
    text_data = file.read()
    
tokenizer = tiktoken.get_encoding("gpt2")
GPT_CONFIG_124M = GPT2_cfg_small.copy()
GPT_CONFIG_124M.update({"context_length": 256})

total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))
print("Characters:", total_characters)
print("Tokens:", total_tokens)
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)
val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

Characters: 20479
Tokens: 5145


In [3]:
# Solution of exercise 5.4
checkpoint = torch.load("../data/model_and_optimizer.pth", map_location=device)
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

num_epochs = 1
start_context = "Every effort moves you"

train_losses, val_losses, tokens_seen = train_model_simple(model, train_loader, val_loader, optimizer, device, 
                                                            num_epochs=num_epochs, eval_freq=5, eval_iter=5, start_context=start_context, tokenizer=tokenizer)

Ep 1 (Step 000000): Train loss 0.281, Val loss 6.564
Ep 1 (Step 000005): Train loss 0.363, Val loss 6.576
Every effort moves you?"     I glanced after him, struck by his last word. Victor Grindle was, in fact, becoming the man of the moment--as Jack himself, one might put it, had been the man of the hour. The
